# AutoRA Workflow

Generated by AutoRA Workflow Editor on 2026-08-16T20:26:25.617Z

## 1. Install dependencies

In [ ]:
%pip install autora-theorist-darts==1.1.0 autora-core==5.0.3 autora-synthetic==2.2.0 autora-experimentalist-falsification==2.2.0 autora-theorist-bms==1.0.6

## 2. Imports

In [ ]:
from autora.state import on_state, Delta, estimator_on_state, StandardState
from autora.variable import VariableCollection
from autora.experimentalist.random import pool as random_pooler, sample as random_sampler
from autora.experiment_runner.synthetic.economics.expected_value_theory import expected_value_theory
from autora.theorist.darts.regressor import DARTSRegressor
from autora.experimentalist.falsification import pool as falsification_pooler, sample as falsification_sampler
from autora.experiment_runner.synthetic.psychology.q_learning import q_learning
from autora.theorist.bms.regressor import BMSRegressor

import pandas as pd

## 3. Component definitions

In [ ]:
# Random Pooler
@on_state()
def random_pooler_on_state(variables: VariableCollection) -> Delta:
    return Delta(conditions=random_pooler(variables, num_samples=5, replace=True))

In [ ]:
# Random Sampler
@on_state()
def random_sampler_on_state(conditions: pd.DataFrame, num_samples: int = 1) -> Delta:
    return Delta(conditions=random_sampler(conditions=conditions, num_samples=num_samples, replace=False))

In [ ]:
# Expected Value Theory (Synthetic, Economics)
runner = expected_value_theory(choice_temperature=0.1, value_lambda=0.5, resolution=10, minimum_value=-1, maximum_value=1)

@on_state()
def expected_value_theory_on_state(conditions: pd.DataFrame) -> Delta:
    return Delta(experiment_data=runner.run(conditions=conditions, added_noise=0.01))

In [ ]:
# DARTS Regressor
darts_regressor_on_state = estimator_on_state(DARTSRegressor(batch_size=64, num_graph_nodes=2, output_type="real", classifier_weight_decay=0.01, darts_type="original", param_updates_per_epoch=10, param_updates_for_sampled_model=100, param_learning_rate_max=0.025, param_learning_rate_min=0.01, param_momentum=0.9, arch_updates_per_epoch=1, arch_learning_rate_max=0.003, arch_weight_decay=0.0001, arch_weight_decay_df=0.0003, arch_weight_decay_base=0, arch_momentum=0.9, fair_darts_loss_weight=1, max_epochs=10, grad_clip=5, primitives=["none", "add", "subtract", "linear", "linear_logistic", "linear_relu"], train_classifier_coefficients=False, train_classifier_bias=False, sampling_strategy="max"))

In [ ]:
# Falsification Pooler
@on_state()
def falsification_pooler_on_state(variables: VariableCollection) -> Delta:
    return Delta(conditions=falsification_pooler(variables, num_samples=100, training_epochs=1000, optimization_epochs=1000, training_lr=0.001, optimization_lr=0.001, limit_offset=0, limit_repulsion=0, plot=False))

In [ ]:
# Falsification Sampler
@on_state()
def falsification_sampler_on_state(conditions: pd.DataFrame, num_samples: int = 1) -> Delta:
    return Delta(conditions=falsification_sampler(conditions=conditions, num_samples=num_samples, training_epochs=1000, training_lr=0.001, plot=False))

In [ ]:
# Q-Learning (Synthetic, Psychology)
runner = q_learning(learning_rate=0.2, decision_noise=3, n_actions=2, forget_rate=0, perseverance_bias=0, correlated_reward=False)

@on_state()
def q_learning_on_state(conditions: pd.DataFrame) -> Delta:
    return Delta(experiment_data=runner.run(conditions=conditions, return_choice_probabilities=False))

In [ ]:
# BMS Regressor
bms_regressor_on_state = estimator_on_state(BMSRegressor(epochs=1500))

## 4. Run the workflow

In [ ]:
# Variables are governed by the experiment runner defined above
assert runner.variables is not None
variables = runner.variables

# Initialize state
state = StandardState(variables=variables)

# Experiment loop (2 cycles)
for i in range(2):
    # Experiment loop (10 cycles)
    for i in range(10):
        print(f'Cycle {i}')

        # Random Pooler
        state = random_pooler_on_state(state)

        # Random Sampler
        state = random_sampler_on_state(state, num_samples=1)

        # Expected Value Theory (Synthetic, Economics)
        state = expected_value_theory_on_state(state)

        # DARTS Regressor
        state = darts_regressor_on_state(state)


    # Experiment loop (1 cycles)
    for i in range(1):
        print(f'Cycle {i}')

        # Falsification Pooler
        state = falsification_pooler_on_state(state)

        # Falsification Sampler
        state = falsification_sampler_on_state(state)

        # Q-Learning (Synthetic, Psychology)
        state = q_learning_on_state(state)

        # BMS Regressor
        state = bms_regressor_on_state(state)



print("Workflow completed!")
state